# Aspect-Based Sentiment Analysis cho đánh giá điện thoại (UIT-ViSFD)

Notebook này **chỉ điều phối và trực quan hóa**. Toàn bộ logic nằm trong gói `src/absa`. Chạy được cả trên máy cá nhân lẫn **Google Colab** (*Runtime → Run all*).

## Quy trình (Run all chạy từ trên xuống dưới)

| Giai đoạn | Mục | Làm gì | Công tắc |
|---|---|---|---|
| **A. EDA** | 1–2 | Đọc dữ liệu, trực quan hóa nhãn, độ dài, xu hướng viết | – |
| **B. Xem xét tiền xử lý** | 3.1–3.4 | Minh họa từng bước, tác động lên từ vựng, **ablation** từng bước | `RUN_ABLATION` |
| **C. Chạy tiền xử lý** | 3.5 | Xử lý toàn bộ train/dev, lưu CSV | – |
| **D. Biểu diễn thưa × mô hình** | 4–5 | Nhiều loại sparse feature × nhiều mô hình ML × chiến lược | `FEATURES`, `MODELS`, `STRATEGIES` |
| **E. Thống kê kết quả** | 5.2–6 | Heatmap, bảng xếp hạng, so sánh chiến lược/biểu diễn/mô hình | – |
| **F. Phân tích lỗi** | 7–8 | Theo khía cạnh, 31 nhãn, confusion matrix, lỗi out-of-fold | `RUN_ERROR_ANALYSIS` |
| **G. Mô hình cuối** | 9–10 | Chọn mô hình tốt nhất, huấn luyện lại train+dev, test ẩn, đóng gói | – |

## Cách nhìn bài toán: 31 nhãn nhị phân (multilabel)

Mỗi bình luận có **31 nhãn nhị phân**: 10 khía cạnh × 3 cảm xúc (`BATTERY#Negative`, `BATTERY#Neutral`, `BATTERY#Positive`, ...) + `OTHERS`. Mỗi nhãn là một bài toán "có/không", nên mọi bộ phân loại nhị phân (LR, SVM, NB, ...) đều dùng được. Đây là chiến lược `multilabel` (Binary Relevance). Notebook **mặc định chạy cả 3 chiến lược** dưới đây để so sánh:

| Chiến lược | Cách làm | Số mô hình con |
|---|---|---|
| `multilabel` | 31 bộ phân loại nhị phân độc lập, rồi giải quyết xung đột (một khía cạnh không thể có 2 cảm xúc) | 31 |
| `joint` | mỗi khía cạnh một bộ phân loại 4 lớp `None/Neg/Neu/Pos` | 11 |
| `two_stage` | phát hiện khía cạnh, rồi phân cực trên các câu có nhắc khía cạnh đó | 21 |

## Cấu hình chạy

Notebook chỉ có **một cấu hình**: 6 biểu diễn × 3 mô hình × 3 chiến lược = 54 cấu hình.

- **6 biểu diễn cốt lõi**: `bow`, `tfidf_word`, `tfidf_word_char` (sparse) và `word2vec`, `fasttext`, `doc2vec` (static embedding).
- **3 mô hình ML cơ bản**: `logistic_regression`, `linear_svm`, `knn`. Không có cây/ensemble (Random Forest, boosting, XGBoost, AdaBoost...) và mạng nơ-ron.
- **3 chiến lược**: `multilabel`, `joint`, `two_stage`.

**Vì sao 6 biểu diễn này?** Mỗi biểu diễn trả lời một câu hỏi:
- `bow` so với `tfidf_word`: trọng số TF-IDF có hơn đếm thô không?
- `tfidf_word` so với `tfidf_word_char`: n-gram ký tự có thêm giá trị so với chỉ n-gram từ không? (Hai TF-IDF này cũng là hai TF-IDF điểm cao nhất ở các lượt thử trước: macro-F1 cặp trên dev với Logistic Regression `tfidf_word_char` 0,534; `tfidf_word` 0,530.)
- `word2vec`, `fasttext`, `doc2vec`: embedding đặc học trên train có thắng được túi từ thưa không? (Lượt thử trước cho Word2Vec/FastText chỉ khoảng 0,43, thấp hơn TF-IDF vì corpus nhỏ.)

Muốn thêm hay bớt thì sửa trực tiếp ba danh sách `FEATURES`, `MODELS`, `STRATEGIES` trong ô cấu hình bên dưới.

**Thời gian**: phần tốn thời gian chủ yếu là ablation, tinh chỉnh top-5 và việc học embedding (Word2Vec/FastText/Doc2Vec) (chưa đo trọn vẹn với bộ này). **Colab miễn phí chỉ có 2 CPU nên chậm hơn khoảng 4–6 lần** (ước tính). Muốn nhanh hơn: đặt `RUN_ABLATION = False` và `RUN_TUNING = False`.

## 0. Thiết lập 

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path


# https://drive.google.com/file/d/14abAEBXzCzEHdmp_kasvmmQRjLhpcvhD/view?usp=drive_link
IN_COLAB = "google.colab" in sys.modules
GDRIVE_ZIP = "14abAEBXzCzEHdmp_kasvmmQRjLhpcvhD"          # <-- Colab: link chia sẻ Google Drive HOẶC file-id của file zip project
CONTENT = Path("/content")


def find_project_root(*starts):
    # Tìm thư mục chứa src/absa (chính nó, cha của nó hoặc thư mục con trực tiếp).
    for start in starts:
        start = Path(start).resolve()
        candidates = [start, *start.parents, *sorted(p for p in start.glob("*") if p.is_dir())]
        for cand in candidates:
            if (cand / "src" / "absa" / "__init__.py").exists():
                return cand
    return None


def pip_install_one_by_one(requirements_file):
    # Cài từng gói một: một gói lỗi chỉ báo cảnh báo, không làm hỏng cả quá trình.
    for line in Path(requirements_file).read_text(encoding="utf-8").splitlines():
        package = line.split("#")[0].strip()
        if not package:
            continue
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", package],
                                capture_output=True, text=True)
        print(("OK       " if result.returncode == 0 else "CẢNH BÁO ") + package)


if IN_COLAB:
    root = find_project_root(CONTENT)
    if root is None:
        assert GDRIVE_ZIP, "Hãy dán link/file-id Google Drive của file zip vào GDRIVE_ZIP ở trên."
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

        zip_path = CONTENT / "project.zip"
        if GDRIVE_ZIP.startswith("http"):
            gdown.download(GDRIVE_ZIP, str(zip_path), quiet=False, fuzzy=True)
        else:
            gdown.download(id=GDRIVE_ZIP, output=str(zip_path), quiet=False)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(CONTENT)
        root = find_project_root(CONTENT)
        assert root is not None, "Không tìm thấy src/absa sau khi giải nén - kiểm tra lại file zip."
    os.chdir(root)
    pip_install_one_by_one(root / "requirements-colab.txt")
    print("Project:", root)
else:
    print("Chạy trên máy cá nhân - bỏ qua bước tải zip.")

## 0.1 Thiết lập môi trường và cấu hình chạy

In [ ]:
import warnings

ROOT = find_project_root(Path.cwd(), CONTENT if IN_COLAB else Path.cwd())
assert ROOT is not None, "Hãy mở notebook trong thư mục project (cần thấy src/absa)."
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))
# Tiến trình con (joblib/loky) cần thấy gói absa khi fit song song các khía cạnh.
os.environ["PYTHONPATH"] = os.pathsep.join([str(SRC), os.environ.get("PYTHONPATH", "")])
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from absa.config import PATHS, PRIMARY_METRIC, SEED
from absa.utils import ensure_dir, save_json, set_seed
from absa.visualization import eda_plots, result_plots, save_figure, set_style

set_seed(SEED)
set_style()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
for d in (PATHS.reports_dir, PATHS.figures_dir, PATHS.models_dir):
    ensure_dir(d)
print("Project root:", ROOT)

import importlib.util

if importlib.util.find_spec("underthesea") is None:
    print("CẢNH BÁO: chưa cài underthesea -> bước chuẩn hóa dấu thanh chỉ dùng NFC, "
          "kết quả tiền xử lý sẽ lệch nhẹ so với bản gốc. Chạy: pip install underthesea")

### Cấu hình chạy

Lưới M x N lưu kết quả **từng dòng** vào CSV nên nếu bị ngắt, chạy lại sẽ tiếp tục từ chỗ dừng.

In [ ]:
RUN_TUNING = True               # tinh chỉnh top-k sau vòng sàng lọc
TOP_K_TUNING = 5
RUN_ABLATION = True             # ablation tiền xử lý (mục 3.4)
RUN_ERROR_ANALYSIS = True       # dự đoán out-of-fold 5 fold trên train (mục 8)
INSPECT_DEV_TEXT = False        # KHÔNG đọc nội dung dev (xem mục 1)
SELECT_STRATEGY = None          # None = chọn mô hình tốt nhất trong mọi chiến lược;
                                # "multilabel" = chỉ chọn trong chiến lược 31 nhãn (để dùng predict_pairs)
N_JOBS = -1

FEATURES = ["bow", "tfidf_word", "tfidf_word_char", "word2vec", "fasttext", "doc2vec"]
MODELS = ["logistic_regression", "linear_svm", "knn"]
STRATEGIES = ("multilabel", "joint", "two_stage")
RESULTS_PATH = PATHS.reports_dir / "ml_grid.csv"
print(f"{len(FEATURES)} biểu diễn x {len(MODELS)} mô hình x {len(STRATEGIES)} chiến lược")
print("Biểu diễn :", FEATURES)
print("Mô hình   :", MODELS)
print("Chiến lược:", STRATEGIES)


## 1. Bài toán và dữ liệu

**Bài toán.** Cho một bình luận về điện thoại, xác định (1) các **khía cạnh** được nhắc tới trong 10 khía cạnh `SCREEN, CAMERA, FEATURES, BATTERY, PERFORMANCE, STORAGE, DESIGN, PRICE, GENERAL, SER&ACC` (+ `OTHERS`), và (2) **cảm xúc** `Positive / Neutral / Negative` cho từng khía cạnh. Một bình luận có thể nhắc nhiều khía cạnh với cảm xúc trái ngược nhau.

**Biểu diễn nhãn.** Ma trận `Y` kích thước `(n, 11)`: mỗi cột một khía cạnh, giá trị `0 = không nhắc, 1 = Negative, 2 = Neutral, 3 = Positive` (OTHERS: 0/1). Từ `Y` suy ra hai nhiệm vụ để đánh giá:
- **Aspect detection**: đa nhãn nhị phân trên 11 khía cạnh.
- **Aspect sentiment**: đa nhãn trên 31 cặp `ASPECT#Polarity`. Một cặp chỉ đúng khi cả khía cạnh lẫn cảm xúc đều đúng.

**Loại bài toán.** Về bản chất là *multi-output multiclass* (11 đầu ra, mỗi đầu ra chọn 1 trong 4 lớp; một khía cạnh không thể có hai cảm xúc cùng lúc). Cách đơn giản nhất là trải thành **31 nhãn nhị phân** và dùng *multilabel classification* (chiến lược `multilabel`): mỗi nhãn một bộ phân loại độc lập, rồi giải quyết xung đột ở bước cuối. Nhược điểm đã biết của Binary Relevance là bỏ qua tương quan giữa các nhãn (Tsoumakas & Katakis, 2007), nên ta so sánh với `joint` và `two_stage`.

**Chính sách dùng các tập (chống rò rỉ thông tin).**

| Tập | Vai trò | Được phép |
|---|---|---|
| Train (7.786) | Học | EDA đầy đủ, thiết kế tiền xử lý, fit vectorizer/mô hình, cross-validation |
| Dev (1.112) | **Validation** | So sánh mô hình, tinh chỉnh, early stopping, ablation. Chỉ code đọc để tính metric; **không** đọc văn bản để đặt luật |
| Test (2.224) | **Hidden test** | Đánh giá **một lần** ở cuối với cấu hình đã chốt. Hiện file đang bị ẩn |

Lý do: test đã bị ẩn nên dev là ước lượng khách quan duy nhất còn lại. Nếu nhìn văn bản dev để đặt luật, điểm dev sẽ bị đẩy lên một cách ảo. Sau khi chọn xong mô hình, dev được gộp vào train để huấn luyện mô hình cuối.

In [ ]:
from absa.data import label_frame, load_splits
from absa.preprocessing import PreparedCorpus, load_lexicons

splits = load_splits(include_test=True)
train, dev, test = splits["train"], splits["dev"], splits["test"]
corpus = PreparedCorpus({name: s.texts for name, s in splits.items() if s is not None})
labels = {name: s.labels for name, s in splits.items() if s is not None}
lexicons = load_lexicons()
print({name: (len(s) if s is not None else "hidden") for name, s in splits.items()})
display(train.frame.head(5))
display(pd.concat([train.frame[["comment"]].head(5), label_frame(train.labels[:5])], axis=1))

## 2. Phân tích khám phá dữ liệu (EDA)

EDA chi tiết **chỉ trên train**. Với dev chỉ so sánh tỉ lệ nhãn ở mức tổng hợp (mục 2.4) để kiểm tra hai tập cùng phân phối.

### 2.1 Tổng quan

In [ ]:
from absa import eda

display(eda.dataset_overview(splits))

### 2.2 Nhãn: phân bố, mất cân bằng, đồng xuất hiện

In [ ]:
counts = eda.aspect_polarity_counts(train.labels)
display(counts)
fig = eda_plots.plot_aspect_polarity(counts); save_figure(fig, "eda_aspect_polarity"); plt.show()
fig = eda_plots.plot_polarity_share(counts); save_figure(fig, "eda_polarity_share"); plt.show()

In [ ]:
fig = eda_plots.plot_labels_per_comment(eda.labels_per_comment(train.labels)); save_figure(fig, "eda_labels_per_comment"); plt.show()
fig = eda_plots.plot_cooccurrence(eda.aspect_cooccurrence(train.labels)); save_figure(fig, "eda_cooccurrence"); plt.show()
display(pd.Series(eda.mixed_sentiment_rate(train.labels), name="%"))

In [ ]:
star_table = eda.star_polarity_crosstab(train.stars, train.labels)
display(star_table)
fig = eda_plots.plot_star_polarity(star_table); save_figure(fig, "eda_star_polarity"); plt.show()

**Nhận xét cần rút ra (điền sau khi chạy):**
- Mất cân bằng mạnh giữa các khía cạnh (GENERAL, PERFORMANCE nhiều; STORAGE rất hiếm) và bên trong khía cạnh (Neutral hiếm) → dùng `class_weight="balanced"`, chọn mô hình theo **macro-F1**.
- Đa số bình luận nhắc từ 2 khía cạnh trở lên, một phần đáng kể có cả cảm xúc tích cực lẫn tiêu cực → phải làm ABSA, không thể dùng một nhãn cảm xúc cho cả câu.
- Số sao chỉ phản ánh xu hướng chung (bình luận 5 sao vẫn có khía cạnh tiêu cực) → không dùng số sao làm nhãn hay đặc trưng.

### 2.3 Văn bản: độ dài và xu hướng viết

In [ ]:
lengths = eda.text_length_frame(train.texts)
display(lengths.describe().T)
fig = eda_plots.plot_length_distribution(lengths); save_figure(fig, "eda_length"); plt.show()

In [ ]:
patterns = eda.text_pattern_report(train.texts, lexicons)
display(patterns)
fig = eda_plots.plot_text_patterns(patterns); save_figure(fig, "eda_text_patterns"); plt.show()

In [ ]:
display(eda.emoji_frequency(train.texts, lexicons, top_n=25))
print(f"Tỉ lệ token không phải âm tiết tiếng Việt chuẩn: {eda.nonstandard_token_rate(train.texts)}%")
display(eda.nonstandard_token_report(train.texts, lexicons, top_n=50))

Bảng trên được dùng để **kiểm tra độ phủ của từ điển một cách có hệ thống**, không để sửa từng câu:
- Token có `in_teencode_lexicon = True` (`ko`, `k`, `dc`, `đt`, `nv`, ...) đã được chuẩn hóa.
- Token còn lại phần lớn là thuật ngữ/thương hiệu (`camera`, `game`, `wifi`, `samsung`, `pubg`) → **giữ nguyên** vì chúng mang thông tin khía cạnh.
- Dạng viết không có trong từ điển vẫn được che chắn bởi luật tổng quát (rút gọn ký tự lặp, `-g` → `-ng`) và bởi TF-IDF n-gram ký tự / FastText subword.

### 2.4 Kiểm tra phân phối dev (chỉ số liệu tổng hợp, không đọc văn bản)

In [ ]:
shift = eda.label_distribution_shift({name: labels.get(name) for name in ("train", "dev")})
shift["diff_pp"] = shift["dev"] - shift["train"]
display(shift)
fig = eda_plots.plot_split_shift(shift[["train", "dev"]]); save_figure(fig, "eda_split_shift"); plt.show()

### 2.5 Từ EDA đến quyết định thiết kế

| Quan sát trên train | Quyết định |
|---|---|
| Nhiều khía cạnh / câu, cảm xúc trái chiều | Mô hình đa đầu ra theo khía cạnh: 31 nhãn nhị phân (`multilabel`) hoặc theo từng khía cạnh (`joint`, `two_stage`) |
| Mất cân bằng lớp mạnh | `class_weight`, metric chính là macro-F1, báo cáo kèm micro-F1 |
| Teencode/viết tắt, tiếng cười, kéo dài ký tự, thiếu `n` trong `-ng` | Luật chuẩn hóa tổng quát + từ điển đã kiểm định (mục 3) |
| Emoji/emoticon mang cảm xúc | Gom về 3 nhãn cảm xúc, có luật dự phòng theo tên Unicode cho emoji chưa gặp |
| Số + đơn vị (`5000mah`, `7tr`, `128gb`) gắn với khía cạnh | Thay bằng token theo **loại đơn vị** thay vì xóa số |
| Xuống dòng thay dấu câu | Đổi xuống dòng thành ranh giới câu |
| Phủ định đảo cảm xúc (`không mượt`) | Gắn phủ định với từ theo sau |

## 3. Tiền xử lý

Mọi bước là **luật tổng quát** (regex theo chính tả tiếng Việt, từ điển có thể đọc và sửa trong `resources/lexicons`), áp dụng giống hệt cho train, dev, test và bình luận mới khi triển khai. Bộ tiền xử lý không học gì từ dữ liệu.

**Cải tiến so với refs:**
- `wrong-word.txt` của refs **bị loại bỏ**: danh sách này xóa cả `pin`, `ok`, `game`, `camera`, `wifi`, `nhanh`, tức đúng những từ mang khía cạnh.
- Stopword của refs chứa `rất`, `quá`, `chưa`, `nhưng`, `được`, `nhanh`, `lâu`, ... nên chỉ dùng **sau khi trừ danh sách từ được bảo vệ** (`protected_words.txt`), và chỉ bật nếu ablation cho thấy có lợi.
- Teencode của refs được lọc các mục sai hoặc mơ hồ trong domain điện thoại (`note`→`chú ý` làm hỏng *Galaxy Note*, `face`→`facebook` làm hỏng *face ID*), rồi bổ sung văn phong review điện thoại.
- Thêm các bước mới: số + đơn vị, emoji có luật dự phòng, emoticon, tiếng cười/khóc, `-g`→`-ng`, xuống dòng, gắn phủ định.

### 3.1 Minh họa từng bước trên một câu

In [ ]:
from absa.preprocessing import TextPreprocessor

demo = "Pin trâu 5000mAh 👍👍, chơi game ko lag tí nào =)) nhưng cam chụp đêm hơi mờ\nGiá 7tr5 khôg rẻ lắm, nv tư vấn nhiệt tìnhhhh"
display(TextPreprocessor("classic").trace(demo))
print("classic ->", TextPreprocessor("classic").process(demo))


### 3.2 Áp dụng cho các tập và đo tác động lên từ vựng

In [ ]:
classic_train = corpus.get("classic", "train")
corpus.get("classic", "dev")
impact = eda.preprocessing_impact(train.texts, classic_train)
display(impact)
fig = eda_plots.plot_preprocessing_impact(impact); save_figure(fig, "prep_vocab_impact"); plt.show()
sample = pd.DataFrame({"raw": train.texts[:8], "classic": classic_train[:8]})
display(sample)

### 3.3 Từ đặc trưng theo khía cạnh sau tiền xử lý (chi²): kiểm tra token tạo ra có ý nghĩa

In [ ]:
for aspect in ["BATTERY", "PERFORMANCE", "PRICE"]:
    terms = eda.top_terms_by_aspect(classic_train, train.labels, aspect, top_n=10)
    fig = eda_plots.plot_top_terms(terms, aspect); save_figure(fig, f"prep_top_terms_{aspect}"); plt.show()

### 3.4 Ablation: mỗi bước đóng góp bao nhiêu?

Cố định TF-IDF + Logistic Regression, chỉ thay cấu hình tiền xử lý và đo trên dev. Đây là phần **motivation định lượng** cho từng bước (yêu cầu 4.2 của đề).

In [ ]:
from absa.training import cumulative_variants, leave_one_out_variants, run_preprocessing_ablation

if RUN_ABLATION:
    ablation_cum = run_preprocessing_ablation(
        train.texts, train.labels, dev.texts, dev.labels, cumulative_variants(), n_jobs=N_JOBS
    )
    ablation_loo = run_preprocessing_ablation(
        train.texts, train.labels, dev.texts, dev.labels, leave_one_out_variants(), n_jobs=N_JOBS
    )
    ablation_cum.to_csv(PATHS.reports_dir / "ablation_cumulative.csv")
    ablation_loo.to_csv(PATHS.reports_dir / "ablation_leave_one_out.csv")
    cols = [PRIMARY_METRIC, "pair_f1_micro", "aspect_f1_macro", "vocab_size"]
    display(ablation_cum[cols]); display(ablation_loo[cols])
    fig = result_plots.plot_ablation(ablation_cum); save_figure(fig, "ablation_cumulative"); plt.show()
    fig = result_plots.plot_ablation(ablation_loo); save_figure(fig, "ablation_leave_one_out"); plt.show()

**Cách đọc:** dòng `- X` là bỏ bước X khỏi `classic`; dòng `+ X` là thêm bước X (chưa có trong `classic`). Điểm *giảm* khi bỏ một bước nghĩa là bước đó có ích.

**Kết quả lượt chạy đầu tiên (đo với chiến lược `joint`, trước khi sửa lỗi gắn phủ định qua ranh giới câu; dùng để chốt cấu hình, chạy lại với `multilabel` và bản đã sửa có thể lệch nhẹ):** đọc cột `pair_f1_macro` để biết bước nào đóng góp dương. Chênh lệch dưới ~0,5 điểm nằm trong mức nhiễu của tập dev 1.112 câu, không nên kết luận từ đó.


### 3.5 Chạy tiền xử lý cho toàn bộ dữ liệu và lưu kết quả

Giai đoạn C: xử lý toàn bộ train/dev bằng profile `classic` (profile của các biểu diễn sparse) và lưu CSV để xem lại hoặc tải về. Kết quả được cache nên lần chạy sau gần như tức thì; sửa từ điển hoặc luật tiền xử lý thì cache tự bị vô hiệu.

In [ ]:
processed_dir = ensure_dir(PATHS.outputs_dir / "processed")
for split_name in ("train", "dev"):
    frame = pd.DataFrame({
        "comment": splits[split_name].texts,
        "classic": corpus.get("classic", split_name),
        "label": splits[split_name].frame["label"].to_numpy(),
    })
    frame.to_csv(processed_dir / f"{split_name}_classic.csv", index=False, encoding="utf-8-sig")
    print(f"{split_name}: {len(frame)} câu -> {processed_dir / (split_name + '_classic.csv')}")
display(pd.DataFrame({"raw": train.texts[:3], "classic": corpus.get("classic", "train")[:3]}))
print("Số câu rỗng sau tiền xử lý (train):", sum(not t.strip() for t in corpus.get("classic", "train")))

## 4. Biểu diễn văn bản (N cách)

Ý tưởng chi tiết từng biểu diễn: `docs/feature_representations.md`. Project dùng hai họ:
- **Sparse** (trọng tâm của phần này): `registry` có 8 biểu diễn (`bow`, `tfidf_word`, `tfidf_char`, `tfidf_word_char` và các biến thể `bow_binary`, `tfidf_word_uni`, `tfidf_word_13`, `tfidf_char_cross`). **Notebook mặc định chạy 6 biểu diễn cốt lõi: `bow`, `tfidf_word`, `tfidf_word_char` (sparse) và `word2vec`, `fasttext`, `doc2vec` (static embedding).** Muốn thử thêm thì sửa danh sách `FEATURES`.
- **Static embedding** (học trên train): Word2Vec, FastText (subword), Doc2Vec.

Mọi biểu diễn được fit **chỉ trên train** (vectorizer nằm trong pipeline).


In [ ]:
from absa.features import feature_catalog

display(feature_catalog())

## 5. Mô hình học máy truyền thống (M mô hình)

`ABSAClassifier` biến **bất kỳ** bộ phân loại nhị phân/đa lớp nào thành bộ phân loại ABSA theo 3 chiến lược:
- `multilabel` (mặc định): 31 bộ phân loại nhị phân độc lập (Binary Relevance). Dự đoán thô là ma trận 31 cột; khi một khía cạnh bị bật nhiều cảm xúc thì chọn cảm xúc có điểm tin cậy cao nhất.
- `joint`: mỗi khía cạnh một bộ phân loại 4 lớp `None/Neg/Neu/Pos`.
- `two_stage`: phát hiện khía cạnh (nhị phân), sau đó phân cực 3 lớp chỉ trên các bình luận có nhắc khía cạnh đó.

`check_compatibility` tự loại các cặp không hợp lệ (ví dụ Multinomial NB với embedding có giá trị âm). Embedding đặc được chuẩn hóa `StandardScaler` cho các mô hình nhạy thang đo.

In [ ]:
from absa.models.ml import model_catalog

display(model_catalog())

### 5.1 Lưới M x N (sàng lọc với tham số mặc định), đánh giá trên dev

In [ ]:
from absa.training import ExperimentRunner

runner = ExperimentRunner(corpus, labels, results_path=RESULTS_PATH, n_jobs=N_JOBS)
grid_results = runner.run_grid(FEATURES, MODELS, STRATEGIES)
print(f"{(grid_results['status'] == 'ok').sum()} cấu hình thành công, "
      f"{(grid_results['status'] == 'failed').sum()} lỗi")
display(runner.leaderboard(top=15))

In [ ]:
results = runner.results
fig = result_plots.plot_score_heatmap(results, PRIMARY_METRIC); save_figure(fig, "ml_heatmap_pair_f1_macro"); plt.show()
fig = result_plots.plot_score_heatmap(results, "aspect_f1_micro"); save_figure(fig, "ml_heatmap_aspect_f1_micro"); plt.show()

In [ ]:
fig = result_plots.plot_leaderboard(runner.leaderboard(top=None)); save_figure(fig, "ml_leaderboard"); plt.show()
fig = result_plots.plot_family_comparison(results); save_figure(fig, "ml_family_comparison"); plt.show()
fig = result_plots.plot_time_vs_score(results); save_figure(fig, "ml_time_vs_score"); plt.show()

### 5.2 Thống kê kết quả: theo chiến lược, biểu diễn và mô hình

Mỗi bảng là trung bình các metric **chỉ trên các cặp có đủ mặt ở mọi nhóm** để so sánh công bằng (bỏ `majority_baseline`).

In [ ]:
from absa.evaluation import summarize_by

if len(STRATEGIES) > 1:
    display(summarize_by(results, "strategy"))
    fig = result_plots.plot_strategy_comparison(results, baseline=STRATEGIES[0]); save_figure(fig, "ml_strategy"); plt.show()
display(summarize_by(results, "feature"))
display(summarize_by(results, "model"))

**Gợi ý đọc kết quả:**
- So sánh **biểu diễn** khi cố định mô hình (một hàng của heatmap, hoặc bảng `summarize_by(results, "feature")`): `bow` vs `tfidf_word` (trọng số), `tfidf_word` vs `tfidf_word_char` (n-gram ký tự có giúp không?), và nhóm sparse vs nhóm embedding đặc (`word2vec`, `fasttext`, `doc2vec`).- So sánh **chiến lược**: nhìn cả `aspect_f1_*` (phát hiện khía cạnh) lẫn `polarity_acc_on_detected` (phân cực). Trong lượt chạy thử nhỏ (2 biểu diễn TF-IDF × 5 mô hình tuyến tính/NB), trung bình macro-F1 cặp là `two_stage` 0,485 ≈ `joint` 0,482 > `multilabel` 0,457: `multilabel` kém hơn ~2,5 điểm, và `aspect_f1_macro` (phát hiện khía cạnh) cũng thấp hơn (0,701 so với 0,736 của `joint`), trong khi `polarity_acc_on_detected` lại cao hơn. Giả thuyết (chưa kiểm chứng): một khía cạnh chỉ được "phát hiện" khi một trong ba bộ phân loại cảm xúc của nó bật, mà mỗi bộ chỉ thấy rất ít mẫu dương. Hãy kiểm tra xem kết quả của bạn có giống không; chênh lệch dưới ~1 điểm nằm trong mức nhiễu của dev 1.112 câu.
- Muốn **dự đoán 31 nhãn** làm mô hình cuối thì đặt `SELECT_STRATEGY = "multilabel"` ở ô cấu hình; mô hình cuối khi đó là cấu hình `multilabel` tốt nhất, dù điểm có thể thấp hơn chiến lược khác.
- Macro-F1 cặp thấp hơn micro-F1 nhiều vì các cặp hiếm (STORAGE, `*#Neutral`).

### 5.3 Tinh chỉnh siêu tham số cho top-k cấu hình (trên dev)

In [ ]:
from absa.training import tune_top_k

if RUN_TUNING:
    tuning_summary = tune_top_k(runner, k=TOP_K_TUNING)
    tuning_summary.to_csv(PATHS.reports_dir / "ml_tuning_summary.csv", index=False)
    display(tuning_summary)

## 6. So sánh tổng hợp và chọn mô hình

Chọn theo `PRIMARY_METRIC` (macro-F1 trên 31 cặp khía cạnh–cảm xúc) trên dev. Bảng dưới gộp kết quả sàng lọc và tinh chỉnh.

In [ ]:
all_results = runner.results.copy()
ok = all_results[all_results["status"] == "ok"].sort_values(PRIMARY_METRIC, ascending=False)
if SELECT_STRATEGY:
    ok = ok[ok["strategy"] == SELECT_STRATEGY]
cols =["tag", "feature", "model", "strategy", "params", PRIMARY_METRIC, "pair_f1_micro",
        "aspect_f1_macro", "aspect_f1_micro", "polarity_acc_on_detected", "fit_sec"]
display(ok[cols].head(15))
all_results.to_csv(PATHS.reports_dir / "all_results.csv", index=False)

best = ok.iloc[0]
print("Cấu hình tốt nhất:", best["feature"], "+", best["model"], f"({best['strategy']})", best["params"])
print(f"{PRIMARY_METRIC} = {best[PRIMARY_METRIC]:.4f} | pair_f1_micro = {best['pair_f1_micro']:.4f}")


## 7. Đánh giá chi tiết mô hình tốt nhất trên dev

In [ ]:
from absa.evaluation import all_confusion_matrices, pair_report, per_aspect_report

Y_dev_pred = runner.predictions.get(best["run_id"])
if Y_dev_pred is None:  # Kết quả nạp từ CSV ở phiên trước: chạy lại cấu hình này.
    import json
    runner.run(best["feature"], best["model"], best["strategy"], json.loads(best["params"]), tag=best["tag"])
    Y_dev_pred = runner.predictions[best["run_id"]]

aspect_report = per_aspect_report(dev.labels, Y_dev_pred)
display(aspect_report.round(3))
fig = result_plots.plot_per_aspect(aspect_report, "Mô hình tốt nhất: kết quả theo khía cạnh (dev)"); save_figure(fig, "best_per_aspect"); plt.show()
fig = result_plots.plot_confusion_grid(all_confusion_matrices(dev.labels, Y_dev_pred)); save_figure(fig, "best_confusion_grid"); plt.show()
display(pair_report(dev.labels, Y_dev_pred).round(3))


### 7.1 Dự đoán thô 31 nhãn của chiến lược `multilabel` và mức xung đột

Với `multilabel`, mỗi nhãn được dự đoán độc lập nên một khía cạnh có thể bị bật **nhiều cảm xúc cùng lúc** (vd `BATTERY#Negative` và `BATTERY#Positive`). Bảng dưới đo mức xung đột đó và so điểm **thô** (31 bit, chưa xử lý) với điểm **sau giải mã** (mỗi khía cạnh một cảm xúc).

In [ ]:
import json as _json

from absa.data import count_pair_conflicts, pair_label_names, to_pair_matrix
from absa.evaluation import evaluate_absa, evaluate_pair_bits, pair_report_from_bits

if best["strategy"] == "multilabel":
    pairs_raw = runner.fit_predict_pairs(best["feature"], best["model"], _json.loads(best["params"]))
    pairs_true = to_pair_matrix(dev.labels)
    print("Dự đoán thô:", pairs_raw.shape, "-> 31 nhãn:", pair_label_names()[:3], "...")
    display(pd.Series(count_pair_conflicts(pairs_raw), name="xung đột").to_frame().T)
    compare = pd.DataFrame({
        "31 bit thô (chưa xử lý xung đột)": evaluate_pair_bits(pairs_true, pairs_raw),
        "sau giải mã (1 cảm xúc / khía cạnh)": {k: v for k, v in evaluate_absa(dev.labels, Y_dev_pred).items() if k.startswith("pair_")},
    })
    display(compare.round(4))
    raw_report = pair_report_from_bits(pairs_true, pairs_raw).round(3)
    display(raw_report.sort_values("f1"))
else:
    print("Mô hình tốt nhất không dùng chiến lược multilabel - bỏ qua mục này.")

## 8. Phân tích lỗi (trên dự đoán out-of-fold của train)

Phân tích lỗi cần **đọc văn bản**, nên làm trên train bằng 5-fold cross-validation: mỗi câu được dự đoán bởi mô hình chưa từng thấy nó. Dev vẫn giữ sạch. Chỉ khi đặt `INSPECT_DEV_TEXT = True` (sau khi đã chốt mô hình) mới xem lỗi trên dev.

In [ ]:
import json

from absa.evaluation import build_error_table, error_type_summary, polarity_confusions, sample_errors
from absa.features import get_feature_spec
from absa.training import cross_val_predict_absa

if RUN_ERROR_ANALYSIS:
    profile = get_feature_spec(best["feature"]).profile
    Y_oof = cross_val_predict_absa(
        corpus.get(profile, "train"), train.labels, best["feature"], best["model"],
        best["strategy"], json.loads(best["params"]), n_jobs=N_JOBS,
    )
    from absa.evaluation import evaluate_absa
    display(pd.Series(evaluate_absa(train.labels, Y_oof)).round(4).to_frame("train OOF"))
    summary = error_type_summary(train.labels, Y_oof)
    display(summary)
    fig = result_plots.plot_error_types(summary); save_figure(fig, "error_types"); plt.show()
    display(polarity_confusions(train.labels, Y_oof))
    errors = build_error_table(train.texts, train.labels, Y_oof, corpus.get(profile, "train"))
    errors.to_csv(PATHS.reports_dir / "error_table_train_oof.csv")
    display(sample_errors(errors, kind="polarity", n=8))
    display(sample_errors(errors, aspect="FEATURES", kind="missed", n=8))

**Khung ghi nhận xét lỗi (điền sau khi chạy):**
1. *Nhầm Neutral ↔ Positive/Negative*: câu so sánh tầm giá (`ổn trong tầm giá`) hoặc điều kiện (`pin trâu nếu để độ sáng vừa`).
2. *Bỏ sót khía cạnh ngầm*: khía cạnh được ám chỉ, không nêu tên (`nóng máy` → BATTERY/PERFORMANCE).
3. *Báo thừa GENERAL*: câu khen chung chung trùng từ vựng với khen khía cạnh cụ thể.
4. *Mơ hồ trong nhãn*: chỉ ra các trường hợp chính nhãn gốc còn tranh cãi.

In [ ]:
if INSPECT_DEV_TEXT:
    dev_errors = build_error_table(dev.texts, dev.labels, Y_dev_pred)
    display(sample_errors(dev_errors, n=10))

## 9. Mô hình cuối: huấn luyện lại trên train + dev và đánh giá test ẩn

1. Huấn luyện lại cấu hình đã chọn trên **train + dev** (dev đã xong vai trò chọn mô hình).
2. Nếu có `Test.csv`: đánh giá **một lần**, ghi kết quả, **không quay lại chỉnh sửa** sau khi đã thấy điểm test.

In [ ]:
from absa.training import fit_final_pipeline

final_pipeline = fit_final_pipeline(
    corpus, labels, best["feature"], best["model"], best["strategy"], best["params"],
    fit_splits=("train", "dev"), n_jobs=N_JOBS,
    extra_metadata={"selection_metric": PRIMARY_METRIC, "dev_score": float(best[PRIMARY_METRIC])},
)
display(pd.Series(final_pipeline.metadata, dtype=object).to_frame("value"))


In [ ]:
from absa.evaluation import evaluate_absa

if test is not None and final_pipeline is not None:
    Y_test_pred = final_pipeline.predict(test.texts)
    test_metrics = evaluate_absa(test.labels, Y_test_pred)
    save_json(test_metrics, PATHS.reports_dir / "test_metrics.json")
    display(pd.Series(test_metrics).round(4).to_frame("test"))
    display(per_aspect_report(test.labels, Y_test_pred).round(3))
else:
    print("Test đang bị ẩn – bỏ qua. Khi có Test.csv trong UIT-ViSFD/, chạy lại riêng cell này.")

## 10. Đóng gói và triển khai

`ABSAPipeline` gói *bộ tiền xử lý + biểu diễn + mô hình + metadata* vào **một file**. Demo:

```bash
streamlit run app/streamlit_app.py
```

In [ ]:
from absa.inference import ABSAPipeline

MODEL_PATH = PATHS.models_dir / "best_ml_pipeline.joblib"
if final_pipeline is not None:
    final_pipeline.save(MODEL_PATH)
    loaded = ABSAPipeline.load(MODEL_PATH)
    new_reviews = [
        "Máy đẹp, pin trâu, chơi liên quân mượt. Tuy nhiên camera chụp tối hơi nhòe.",
        "Mới mua 2 tuần đã bị lỗi cảm ứng, mang ra bảo hành thì nhân viên thái độ kém :((",
        "Giá 5tr mà cấu hình vậy là ok rồi, loa hơi nhỏ",
        "sp tốt, giao hàng nhanh, đóng gói cẩn thận 👍👍",
    ]
    display(loaded.predict_frame(new_reviews))
    if loaded.metadata.get("strategy") == "multilabel":
        pairs_new = loaded.predict_pairs(new_reviews)  # 31 nhãn nhị phân thô
        display(pairs_new.T.loc[lambda d: d.sum(axis=1) > 0].rename(columns=lambda i: f"review {i}"))
    summary = loaded.summarize(dev.texts)  # minh họa tổng hợp trên một tập bình luận
    display(summary)
    fig = result_plots.plot_product_summary(summary); save_figure(fig, "deploy_product_summary"); plt.show()

## 11. Kết luận và hướng phát triển

- **Tóm tắt kết quả**: cấu hình tốt nhất (biểu diễn sparse nào, mô hình nào, chiến lược nào), tác dụng của bigram/trigram và n-gram ký tự, so sánh `multilabel`/`joint`/`two_stage`.
- **Tiền xử lý**: những bước có đóng góp dương rõ ràng theo ablation.
- **Hạn chế**: khía cạnh hiếm (STORAGE), nhãn Neutral, khía cạnh ngầm; Binary Relevance bỏ qua tương quan giữa các nhãn.
- **Hướng phát triển**: Classifier Chains (khai thác tương quan nhãn); tinh chỉnh ngưỡng quyết định theo từng nhãn trên dev; tăng cường dữ liệu cho khía cạnh hiếm.


## 12. Xuất kết quả (tải về từ Colab)

Gói bảng kết quả, hình vẽ, dữ liệu đã tiền xử lý và mô hình thành một file zip. Trên Colab, file được tải về máy; nhớ làm bước này trước khi runtime bị ngắt vì `/content` là bộ nhớ tạm.

In [ ]:
export_path = ROOT / "absa_outputs.zip"
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PATHS.outputs_dir.rglob("*")):
        if path.is_file() and "cache" not in path.relative_to(PATHS.outputs_dir).parts:
            archive.write(path, path.relative_to(PATHS.outputs_dir))  # bỏ cache (nặng, tính lại được)
print("Đã tạo:", export_path, f"({export_path.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(str(export_path))